# Notebook 3 - Data Quality Assessment
Sprint 4 - EDA

In this notebook I check the quality of my dataset before analysing it. I look for missing values, duplicate records and invalid values. The goal is only to find and document the problems. I will not fix them here, because cleaning is part of the next sprint.

**Dataset:** Telco Customer Churn (Kaggle, 7043 rows and 21 columns)

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")
print("Shape:", df.shape)

Shape: (7043, 21)


## Part 1 - Missing Values

### 1.1 Number and percentage of missing values

**Concept:** A missing value is a place where data should be present but is not. In pandas the real missing value is NaN, and `isnull().sum()` counts it for every column.

**Example (business):** A bank table has 1000 customers and the Income column is empty for 150 of them. That is 15% missing. If a loan model ignores this, it may reject many good customers or crash. The percentage matters more than the count, because 15 missing values in 20 rows is a disaster while 15 in 1 million rows is not.

In [2]:
missing_count = df.isnull().sum()
missing_percent = (df.isnull().sum() / len(df) * 100).round(2)

missing_table = pd.DataFrame({"missing_count": missing_count, "missing_percent": missing_percent})
print(missing_table.sort_values("missing_count", ascending=False).head(10))
print()
print("Total missing cells:", df.isnull().sum().sum())

                 missing_count  missing_percent
customerID                   0              0.0
gender                       0              0.0
SeniorCitizen                0              0.0
Partner                      0              0.0
Dependents                   0              0.0
tenure                       0              0.0
PhoneService                 0              0.0
MultipleLines                0              0.0
InternetService              0              0.0
OnlineSecurity               0              0.0

Total missing cells: 0


**What the code does:** `isnull()` marks each cell as True if it is NaN, and `sum()` counts the True values for each column. I divided by the number of rows to get the percentage and put both in one table.

**Why this method was used:** Counting by column shows which features are affected and how badly. The percentage tells if the problem is small or serious.

**What the output means:** Every column shows 0 missing values and the total missing cells is 0.

**Insights:** On the surface the dataset looks complete. But in Notebook 2 I saw that `TotalCharges` has blank spaces, and `isnull()` cannot see them because a blank space is a text value and not NaN. So this result is incomplete and I must look for hidden missing values.

**Effect on the ML pipeline:** If I trusted this output, I would skip missing value treatment and the model would fail or behave wrongly later. A data quality check must always go beyond `isnull()`.

### 1.2 Hidden missing values (blank strings)

**Concept:** Sometimes missing data is stored as an empty string, spaces or words like NA or unknown. pandas does not treat these as missing, so they have to be found by checking the text values.

**Example (AI/ML):** A survey file has the value " " in an age column. The column becomes text, and the mean age cannot be calculated. After replacing the blanks with NaN and converting to numbers, the column works normally.

In [3]:
# find blank or space-only values in every text column
text_cols = df.select_dtypes(exclude="number").columns

blank_counts = {}
for col in text_cols:
    blank_counts[col] = (df[col].astype(str).str.strip() == "").sum()

blank_series = pd.Series(blank_counts)
print(blank_series[blank_series > 0])

TotalCharges    11
dtype: int64


In [4]:
# look at those rows
blank_rows = df[df["TotalCharges"].astype(str).str.strip() == ""]
blank_rows[["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Contract", "Churn"]]

,customerID,tenure,MonthlyCharges,TotalCharges,Contract,Churn
488,4472-LVYGI,0,52.55,,Two year,No
753,3115-CZMZD,0,20.25,,Two year,No
936,5709-LVOEQ,0,80.85,,Two year,No
1082,4367-NUYAO,0,25.75,,Two year,No
1340,1371-DWPAZ,0,56.05,,Two year,No
3331,7644-OMVMY,0,19.85,,Two year,No
3826,3213-VVOLG,0,25.35,,Two year,No
4380,2520-SGTTA,0,20.00,,Two year,No
5218,2923-ARZLG,0,19.70,,One year,No
6670,4075-WKNIU,0,73.35,,Two year,No


**What the code does:** The first cell goes through every non-numeric column, removes spaces from each value with `strip()` and counts the values that become empty. The second cell displays the rows of `TotalCharges` that are blank.

**Why this method was used:** A loop over all text columns checks every column in one go, so I do not miss a column. Looking at the actual rows helps me understand why they are blank.

**What the output means:** Only `TotalCharges` has blank values, 11 rows. All of them have `tenure` equal to 0, and their `MonthlyCharges` is a normal value.

**Insights:** The blanks are not random. These are brand new customers who joined but have not completed a billing month, so the total charge does not exist yet. The missing values are therefore explained by `tenure`, which means the missingness has a pattern. This is only 11 rows, which is 0.16% of the data.

**Effect on the ML pipeline:** I will convert `TotalCharges` to numbers (the blanks become NaN) and then decide how to treat them. Since the true total for a new customer is logically 0, filling with 0 is one option, and dropping the 11 rows is another. I will decide in the preprocessing sprint and not here.

### 1.3 Missing values by column and missing value patterns

**Concept:** After converting hidden blanks to real NaN, I can count missing values by column and check if the missing rows share a pattern (for example, all belong to the same group).

**Example (business):** In a hospital table, blood sugar is missing only for patients who were not tested. If the missingness follows a group, it carries information, and filling it with the average may hide that.

In [5]:
df_check = df.copy()
df_check["TotalCharges"] = pd.to_numeric(df_check["TotalCharges"], errors="coerce")

print("Missing values by column after conversion:")
print(df_check.isnull().sum()[df_check.isnull().sum() > 0])
print()

missing_rows = df_check[df_check["TotalCharges"].isnull()]
print("Tenure values in missing rows:", missing_rows["tenure"].unique())
print("Churn in missing rows:")
print(missing_rows["Churn"].value_counts())
print()
print("Customers with tenure = 0 in full data:", (df_check["tenure"] == 0).sum())

Missing values by column after conversion:
TotalCharges    11
dtype: int64

Tenure values in missing rows: [0]
Churn in missing rows:
Churn
No    11
Name: count, dtype: int64

Customers with tenure = 0 in full data: 11


**What the code does:** `pd.to_numeric` with `errors=coerce` turns every value that cannot become a number into NaN. After that I count missing values per column, then look at the `tenure` and `Churn` values of the missing rows and compare with the number of customers who have tenure 0 in the whole data.

**Why this method was used:** `errors=coerce` is a quick way to convert the column and expose the bad values at the same time. I worked on a copy (`df_check`) so the original data stays unchanged.

**What the output means:** After conversion `TotalCharges` has 11 missing values and no other column has any. All the missing rows have `tenure` 0, and the count of customers with tenure 0 in the full data is also 11, so every customer with tenure 0 has a missing `TotalCharges`.

**Insights:** The pattern is complete: missing `TotalCharges` happens exactly when `tenure` is 0. This is a structural missing value, not a random one. Also the `Churn` column of these rows is shown above, and it can be used to see whether these new customers left.

**Effect on the ML pipeline:** Because the missingness is fully explained by another feature, a simple rule based fix is possible instead of a statistical guess. It also shows that the mean or median should not be used blindly to fill this column.

## Part 2 - Duplicate Records

### 2.1 Completely duplicated rows

**Concept:** A duplicate record is a row that appears more than once. A completely duplicated row has the same value in every column. `duplicated()` marks the repeated rows and `duplicated().sum()` counts them.

**Example (numerical):** If an orders table has 1000 rows and 40 of them are exact copies, then total sales are overcounted. A model trained on that table sees those 40 orders twice and gives them extra weight.

In [6]:
print("Completely duplicated rows:", df.duplicated().sum())
print("Percentage:", round(df.duplicated().sum() / len(df) * 100, 2), "%")

Completely duplicated rows: 0
Percentage: 0.0 %


**What the code does:** `df.duplicated()` returns True for every row that is an exact copy of an earlier row, and `sum()` counts them.

**Why this method was used:** It is the standard first check for repeated rows, and it compares all columns together.

**What the output means:** There are 0 completely duplicated rows.

**Insights:** No row is copied exactly. This is expected here, because every row has a different `customerID`, which makes every row different even if the customers have the same details.

**Effect on the ML pipeline:** No deduplication step is needed for exact copies. But this check is not enough on its own, because of the ID column, so I check for potential duplicates next.

### 2.2 Duplicates in the identifier and potential duplicate records

**Concept:** A potential duplicate is a record that may describe the same entity even if some columns differ, for example the same customer with two different ids. A simple way is to ignore the identifier column and check if all other columns match.

**Example (business):** A CRM has two entries for the same person, with different customer numbers but the same name, phone and address. They look like two customers but are one, and this inflates the customer count.

In [7]:
# duplicate ids
print("Duplicate customerID values:", df["customerID"].duplicated().sum())

# potential duplicates: same values in all columns except customerID
cols_without_id = [c for c in df.columns if c != "customerID"]
potential_dups = df.duplicated(subset=cols_without_id, keep=False)

print("Rows that share all feature values with another row:", potential_dups.sum())
print("Extra copies (keeping first):", df.duplicated(subset=cols_without_id).sum())

Duplicate customerID values: 0
Rows that share all feature values with another row: 42
Extra copies (keeping first): 22


In [8]:
# view some of them, sorted so identical rows come together
df[potential_dups].sort_values(cols_without_id).head(10)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
6491,9728-FTTVZ,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,69.20,69.2,Yes
6764,7660-HDPJV,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,69.20,69.2,Yes
4495,4702-IOQDC,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.10,70.1,Yes
6267,0328-GRPMV,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.10,70.1,Yes
5522,2619-WFQWU,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Mailed check,70.15,70.15,Yes
5759,9985-MWVIX,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Mailed check,70.15,70.15,Yes
542,2866-IKBTM,Female,0,No,No,1,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Month-to-month,No,Mailed check,19.55,19.55,No
1491,8605-ITULD,Female,0,No,No,1,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Month-to-month,No,Mailed check,19.55,19.55,No
5170,7721-DVEKZ,Female,0,No,No,1,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Month-to-month,No,Mailed check,19.65,19.65,No
6774,0970-QXPXW,Female,0,No,No,1,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Month-to-month,No,Mailed check,19.65,19.65,No


**What the code does:** First I count repeated `customerID` values. Then I use `duplicated` with `subset` to compare all columns except the id. With `keep=False` every member of a repeated group is marked, so I can see all the copies, and without it only the extra copies are counted.

**Why this method was used:** The id is always unique, so a normal duplicate check cannot find repeated customers. Ignoring the id lets me find rows that have identical details.

**What the output means:** There are no duplicate ids. Some rows have exactly the same values in all 20 other columns as another row, and the number of such extra copies is printed above (a small count compared to 7043).

**Insights:** These are only potential duplicates. The dataset has 19 features and most are Yes/No type with a few numbers, so two different customers can match by chance, especially for short tenure and common plans. Because the ids are different, I cannot say that they are real duplicates without more information, so I should not delete them automatically.

**Effect on the ML pipeline:** Removing real duplicates avoids giving some customers double weight and avoids leakage between training and test sets. But removing rows that are not true duplicates loses valid data. I will document this and decide in the preprocessing sprint, after checking whether `MonthlyCharges` and `TotalCharges` are also identical.

## Part 3 - Invalid Values

### 3.1 Negative and impossible values

**Concept:** Some columns can never be negative (age, price, quantity) or have a natural limit (a percentage cannot go above 100). Values outside the possible range are invalid and usually come from entry or system errors.

**Example (numerical):** An age column with the value -5 or 250 is impossible. If the model learns from it, the mean and the scaling are distorted.

In [9]:
num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]

print("Negative values:")
for col in num_cols:
    print(col, ":", (df_check[col] < 0).sum())

print()
print("Zero values:")
for col in num_cols:
    print(col, ":", (df_check[col] == 0).sum())

print()
print(df_check[num_cols].agg(["min", "max"]))

Negative values:
tenure : 0
MonthlyCharges : 0
TotalCharges : 0

Zero values:
tenure : 11
MonthlyCharges : 0
TotalCharges : 0

     tenure  MonthlyCharges  TotalCharges
min       0           18.25          18.8
max      72          118.75        8684.8


**What the code does:** I count negative and zero values in the three numerical columns (using the converted copy) and print the minimum and maximum of each.

**Why this method was used:** Billing amounts and months of tenure cannot be negative, so a simple comparison with 0 finds impossible values. Min and max give a quick range check.

**What the output means:** There are no negative values. `tenure` has 11 zeros, which are the new customers. `MonthlyCharges` and `TotalCharges` have no zeros. The minimum and maximum look realistic (`tenure` 0 to 72, `MonthlyCharges` about 18 to 119).

**Insights:** No impossible numbers were found. The zero `tenure` values are valid, but they are connected to the missing `TotalCharges` problem.

**Effect on the ML pipeline:** No treatment is needed for negative values. The zero tenure customers need a decision because a feature like average spend per month would divide by zero if I engineer it later.

### 3.2 Consistency between columns

**Concept:** Some values are only valid in combination with other columns. For example, if a customer has no internet, then add-on services like online security cannot be active.

**Example (business):** An airline record that shows a passenger as checked in but not booked is logically impossible, even if each value alone looks fine.

In [10]:
internet_dependent = ["OnlineSecurity", "OnlineBackup", "DeviceProtection",
                      "TechSupport", "StreamingTV", "StreamingMovies"]

no_internet = df[df["InternetService"] == "No"]
print("Customers with no internet service:", len(no_internet))
print()
for col in internet_dependent:
    print(col, "->", no_internet[col].unique())

print()
no_phone = df[df["PhoneService"] == "No"]
print("Customers with no phone service:", len(no_phone))
print("MultipleLines values for them:", no_phone["MultipleLines"].unique())

Customers with no internet service: 1526

OnlineSecurity -> <ArrowStringArray>
['No internet service']
Length: 1, dtype: str
OnlineBackup -> <ArrowStringArray>
['No internet service']
Length: 1, dtype: str
DeviceProtection -> <ArrowStringArray>
['No internet service']
Length: 1, dtype: str
TechSupport -> <ArrowStringArray>
['No internet service']
Length: 1, dtype: str
StreamingTV -> <ArrowStringArray>
['No internet service']
Length: 1, dtype: str
StreamingMovies -> <ArrowStringArray>
['No internet service']
Length: 1, dtype: str

Customers with no phone service: 682
MultipleLines values for them: <ArrowStringArray>
['No phone service']
Length: 1, dtype: str


**What the code does:** I select customers with no internet and print the values of the six internet add-on columns for them. I do the same for MultipleLines among customers with no phone service.

**Why this method was used:** These are business rules between columns. A single column check cannot find such errors.

**What the output means:** For customers with no internet service, all six add-on columns have only one value, No internet service. For customers with no phone service, `MultipleLines` has only No phone service.

**Insights:** The data is consistent, there is no case of an add-on active without internet or multiple lines without phone. It also shows that No internet service and No phone service are a third label in columns that look like Yes/No.

**Effect on the ML pipeline:** These extra labels carry the same meaning as No in most cases. During encoding I can either merge them into No or keep them as a separate category. Merging reduces the number of columns, so I will test this in the preprocessing sprint.

### 3.3 Incorrect categories and unexpected values

**Concept:** Categorical columns should only contain the allowed labels. Wrong spellings, different cases (Yes and yes) or extra spaces create fake categories.

**Example (AI/ML):** A column has Male, male and M for the same value. The model sees three different groups, and one-hot encoding creates three columns instead of one.

In [11]:
cat_cols = [c for c in df.columns if c not in ["customerID", "tenure", "MonthlyCharges", "TotalCharges"]]

for col in cat_cols:
    print(col, "->", sorted(df[col].unique().tolist()))

gender -> ['Female', 'Male']
SeniorCitizen -> [0, 1]
Partner -> ['No', 'Yes']
Dependents -> ['No', 'Yes']
PhoneService -> ['No', 'Yes']
MultipleLines -> ['No', 'No phone service', 'Yes']
InternetService -> ['DSL', 'Fiber optic', 'No']
OnlineSecurity -> ['No', 'No internet service', 'Yes']
OnlineBackup -> ['No', 'No internet service', 'Yes']
DeviceProtection -> ['No', 'No internet service', 'Yes']
TechSupport -> ['No', 'No internet service', 'Yes']
StreamingTV -> ['No', 'No internet service', 'Yes']
StreamingMovies -> ['No', 'No internet service', 'Yes']
Contract -> ['Month-to-month', 'One year', 'Two year']
PaperlessBilling -> ['No', 'Yes']
PaymentMethod -> ['Bank transfer (automatic)', 'Credit card (automatic)', 'Electronic check', 'Mailed check']
Churn -> ['No', 'Yes']


In [12]:
# check for hidden spacing or case problems in text columns
for col in df.select_dtypes(exclude="number").columns:
    if col in ["customerID", "TotalCharges"]:
        continue
    cleaned = df[col].str.strip().str.lower()
    if cleaned.nunique() != df[col].nunique():
        print("Possible spelling or spacing issue in:", col)
print("Check finished")

Check finished


**What the code does:** The first cell prints the unique labels of every categorical column. The second cell cleans each text column (removes spaces, makes it lowercase) and compares the number of unique values before and after. If it drops, two labels differed only by spacing or case.

**Why this method was used:** Reading the unique values is the direct way to spot a wrong label, and the before-and-after comparison catches small differences that are hard to see by eye.

**What the output means:** All categorical columns contain only expected labels, such as Yes and No, Male and Female, three contract types and four payment methods. `SeniorCitizen` only contains 0 and 1. The second check printed no issue.

**Insights:** There are no spelling, case or spacing problems in the categories. The only oddity is the extra No internet service and No phone service labels discussed above.

**Effect on the ML pipeline:** Encoding can be done directly on these columns without text cleaning. If there had been inconsistent labels, they would first need to be standardised.

### 3.4 Empty strings and null-like values

**Concept:** Missing data is often written as words or symbols such as NA, N/A, null, none, unknown, a dash or a question mark. These are text values, so pandas does not count them as missing.

**Example (business):** A form where users type unknown in the city field. Reports then show unknown as the most popular city.

In [13]:
null_like = ["", " ", "na", "n/a", "nan", "null", "none", "unknown", "-", "?", "missing", "not available"]

found = {}
for col in df.select_dtypes(exclude="number").columns:
    cleaned = df[col].astype(str).str.strip().str.lower()
    count = cleaned.isin(null_like).sum()
    if count > 0:
        found[col] = count

print(found)

{'TotalCharges': np.int64(11)}


**What the code does:** I made a list of common null-like words and symbols. For each text column I clean the values (strip spaces, lowercase) and count how many match the list.

**Why this method was used:** Null-like values come in many forms, so checking against a list is more reliable than checking for one.

**What the output means:** Only `TotalCharges` is found, with 11 matches (the blank spaces). No other column contains empty strings or null-like words. Labels like No and No internet service are real answers and not missing values.

**Insights:** This confirms that the blank `TotalCharges` is the only hidden missing value problem in the data.

**Effect on the ML pipeline:** I have to replace these blanks with NaN first, so that standard imputation methods can see them.

### 3.5 Incorrect data types and invalid dates

**Concept:** A column should have the type that matches its meaning. Amounts should be numbers and dates should be datetime. Wrong types block calculations and make plots fail.

**Example (AI/ML):** If a date column is stored as text, I cannot calculate the number of days between two dates or sort the rows by time.

In [14]:
print("Columns with date or time in the name:",
      [c for c in df.columns if "date" in c.lower() or "time" in c.lower()])
print()

# compare stored type and real meaning
print("TotalCharges stored as:", df["TotalCharges"].dtype, "(should be numeric)")
print("SeniorCitizen stored as:", df["SeniorCitizen"].dtype, "(should be categorical)")
print("SeniorCitizen values:", df["SeniorCitizen"].unique())

Columns with date or time in the name: []

TotalCharges stored as: str (should be numeric)
SeniorCitizen stored as: int64 (should be categorical)
SeniorCitizen values: [0 1]


**What the code does:** The code looks for date or time columns by name and prints the stored data type of `TotalCharges` and `SeniorCitizen` next to what they should be.

**Why this method was used:** The dtype is only what pandas guessed, so I compare it with the real meaning of each column.

**What the output means:** There are no date columns, so invalid dates cannot occur here. `TotalCharges` is stored as text but should be numeric. `SeniorCitizen` is stored as a number but its values are only 0 and 1, so it is categorical.

**Insights:** There are two type problems, and the first one is the cause of the hidden blank values.

**Effect on the ML pipeline:** `TotalCharges` has to be converted to float and `SeniorCitizen` has to be treated as a category, otherwise it could be wrongly scaled as a number.

## Data Quality Summary

In [15]:
summary = pd.DataFrame({
    "check": ["Missing values (isnull)",
              "Hidden missing values (blank TotalCharges)",
              "Completely duplicated rows",
              "Duplicate customerID",
              "Negative values",
              "Wrong category labels",
              "Date columns",
              "Wrong data types"],
    "result": [df.isnull().sum().sum(),
               (df["TotalCharges"].astype(str).str.strip() == "").sum(),
               df.duplicated().sum(),
               df["customerID"].duplicated().sum(),
               (df_check[num_cols] < 0).sum().sum(),
               0,
               0,
               2]
})
summary

,check,result
0,Missing values (isnull),0
1,Hidden missing values (blank TotalCharges),11
2,Completely duplicated rows,0
3,Duplicate customerID,0
4,Negative values,0
5,Wrong category labels,0
6,Date columns,0
7,Wrong data types,2


| Area | Finding | Severity |
|---|---|---|
| Missing values | None found by `isnull()` | - |
| Hidden missing values | 11 blank `TotalCharges` (0.16%), all with tenure 0 | Medium |
| Complete duplicates | 0 | None |
| Potential duplicates | A few rows match on all features except `customerID` | Low (need review) |
| Negative or impossible values | None | None |
| Wrong category labels | None | None |
| Inconsistent columns | None, but extra labels like No internet service exist | Low |
| Wrong data types | `TotalCharges` (text instead of number), `SeniorCitizen` (number instead of category) | Medium |

### Issues to handle in the preprocessing sprint
1. Convert `TotalCharges` to numeric and decide how to treat the 11 customers with tenure 0.
2. Convert `SeniorCitizen` to a categorical column.
3. Review the potential duplicate rows before deciding to remove any.
4. Decide how to treat No internet service and No phone service labels during encoding.

### What I learned
- A dataset can look perfectly complete with `isnull()` and still contain hidden missing values.
- The missing values here have a clear reason (new customers), so the fix should follow the logic of the business and not just a statistic.
- Rows that look duplicated may be different customers, so deleting them without thinking can remove valid data.

Next: `04_Univariate_Analysis.ipynb`